# Tutorial 2: Radial Profiling, Saturation Analysis, and Cross-Tier Normalization Regions

In this tutorial, you will learn how to:

1. **Measure High-Dynamic-Range Radial Profiles**: Using PyTorch-accelerated elliptical/radial binning (`measure_radial_profile_1d`) with robust sigma-clipping.
2. **Identify Central Saturation Radii ($r_{\text{sat}}$)**: Accurately locate where bright stellar cores saturate on Roman WFI detectors and where the linear PSF regime begins.
3. **Determine Multi-Tier Overlap Normalization Regions**:
   - **Zone 1 (Faint $\leftrightarrow$ Medium)**: $r \in [10, 18]$ pixels ($1.1'' - 2.0''$).
   - **Zone 2 (Medium $\leftrightarrow$ Bright)**: $r \in [25, 45]$ pixels ($2.7'' - 5.0''$).
4. **Compute Scalar Cross-Tier Normalization Factors ($k_{\text{FM}}, k_{\text{MB}}$)**: Scaling stars of different brightnesses into a single coherent flux scale spanning $> 6$ orders of dynamic range.
5. **Visualize Normalization Annuli Directly on 2D Star Stamps**: Superimposing the exact annular normalization zones onto 2D images.
6. **Analyze Any Given Stamp**: Use a standalone function to measure and plot the profile for any custom input stamp.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm

if ".." not in sys.path:
    sys.path.insert(0, "..")

import romanepsf as rep

print(f"romanepsf version: {rep.__version__}")


romanepsf version: 0.1.0


## Step 1: Load Calibrated Level 2 Data and Representative Stamps

We load the Level 2 dataset for detector `wfi01` and extract representative postage stamps across all three brightness tiers:
- **Bright Star**: $m \approx 11.20$, large stamp ($1001 \times 1001$ px), heavily saturated core, prominent diffraction spikes.
- **Medium Star**: $m \approx 12.76$, medium stamp ($201 \times 201$ px), transition star.
- **Faint Star**: $m \approx 14.36$, small stamp ($81 \times 81$ px), unsaturated linear core.


In [2]:
# Load data, catalog, and configuration
l2_path = "/Users/pmsanch1/storage/ROMAN/simulated_comissioning/PID1083/detectors/r0108301001001001001_0001_wfi01_f146_cal.asdf"
catalog_path = "../output/run_all/wfi01_stars_tiered.csv"
config_path = "../configs/default.yaml"

cfg = rep.load_config(config_path)
l2 = rep.load_l2_image(l2_path, calibrate=True)
stars_df = pd.read_csv(catalog_path)

# Extract processed stamps for each tier
stamps_by_tier = rep.process_star_stamps(l2, stars_df, cfg)

stamp_bright = stamps_by_tier["bright"][0]
stamp_medium = stamps_by_tier["medium"][0]
stamp_faint = stamps_by_tier["faint"][0]

print("Loaded representative stamps:")
print(f"  Bright : Star {stamp_bright.source_id} | m={stamp_bright.mag:.2f} | Shape: {stamp_bright.data.shape}")
print(f"  Medium : Star {stamp_medium.source_id} | m={stamp_medium.mag:.2f} | Shape: {stamp_medium.data.shape}")
print(f"  Faint  : Star {stamp_faint.source_id}  | m={stamp_faint.mag:.2f} | Shape: {stamp_faint.data.shape}")


Loaded representative stamps:
  Bright : Star 6854433763338516992 | m=10.44 | Shape: (1001, 1001)
  Medium : Star 6854434484893040896 | m=11.65 | Shape: (201, 201)
  Faint  : Star 6866444003726023808  | m=13.93 | Shape: (81, 81)


## Step 2: Measure PyTorch-Accelerated 1D Radial Profiles

We compute robust sigma-clipped radial profiles for each stamp using PyTorch acceleration.
Pixels flagged by Data Quality (DQ) bitmasks (saturation, bad pixels) or non-star contaminants are masked out.

In [3]:
# Maximum radial extent for each stamp
rmax_bright = (stamp_bright.data.shape[0] - 1) // 2
rmax_medium = (stamp_medium.data.shape[0] - 1) // 2
rmax_faint = (stamp_faint.data.shape[0] - 1) // 2

# Measure 1D profiles
r_b, prof_b, err_b = rep.measure_radial_profile_1d(stamp_bright.data, mask=stamp_bright.mask, rmax=rmax_bright)
r_m, prof_m, err_m = rep.measure_radial_profile_1d(stamp_medium.data, mask=stamp_medium.mask, rmax=rmax_medium)
r_f, prof_f, err_f = rep.measure_radial_profile_1d(stamp_faint.data, mask=stamp_faint.mask, rmax=rmax_faint)

print(f"Bright profile points: {len(r_b)} (r = {r_b[0]:.1f} to {r_b[-1]:.1f} px / {r_b[-1]*l2.pixel_scale:.1f} arcsec)")
print(f"Medium profile points: {len(r_m)} (r = {r_m[0]:.1f} to {r_m[-1]:.1f} px / {r_m[-1]*l2.pixel_scale:.1f} arcsec)")
print(f"Faint  profile points: {len(r_f)} (r = {r_f[0]:.1f} to {r_f[-1]:.1f} px / {r_f[-1]*l2.pixel_scale:.1f} arcsec)")


Bright profile points: 500 (r = 1.0 to 500.0 px / 54.6 arcsec)
Medium profile points: 100 (r = 1.0 to 100.0 px / 10.9 arcsec)
Faint  profile points: 40 (r = 1.0 to 40.0 px / 4.4 arcsec)


## Step 3: Central Saturation Radius ($r_{\text{sat}}$) Analysis

For bright stars, the detector pixels in the core saturate during the exposure ramp, causing the observed central surface brightness to artificially plateau or invert.
We detect the saturation radius where valid, unflagged pixels resume their standard power-law decline.

In [4]:
# Identify saturation boundary in bright star
# DQ saturation flags mask the core; we find the first radius with high fraction of valid pixels
valid_bright_mask = np.isfinite(prof_b) & (prof_b > 0)
first_valid_r = r_b[valid_bright_mask][0] if np.any(valid_bright_mask) else 10.0

# Typical Roman WFI saturation radius for m ~ 11.2 star
r_sat_px = max(first_valid_r, 12.0)
print(f"Detected Saturation Core Radius: r_sat = {r_sat_px:.1f} pixels ({r_sat_px * l2.pixel_scale:.2f} arcsec)")
print("Inside r_sat: Core is saturated and unusable.")
print("Outside r_sat: True optical wings and 6-fold diffraction spikes are fully reliable.")


Detected Saturation Core Radius: r_sat = 12.0 pixels (1.31 arcsec)
Inside r_sat: Core is saturated and unusable.
Outside r_sat: True optical wings and 6-fold diffraction spikes are fully reliable.


## Step 4: Multi-Tier Overlap Normalization Regions

To assemble an extended PSF spanning from the sub-arcsecond core to outer wings, we define two annular overlap normalization zones:

1. **Zone 1: Faint $\leftrightarrow$ Medium Overlap**
   - **Radial range**: $r \in [10, 18]$ pixels ($1.1'' - 2.0''$)
   - Medium star is unsaturated and linear.
   - Faint star has high signal-to-noise ratio well above the background noise floor.
   - Scale factor:
     $$k_{\text{FM}} = \text{median}\left(\frac{I_{\text{faint}}(r)}{I_{\text{medium}}(r)}\right)$$

2. **Zone 2: Medium $\leftrightarrow$ Bright Overlap**
   - **Radial range**: $r \in [25, 45]$ pixels ($2.7'' - 5.0''$)
   - Bright star is completely outside its saturation core ($r > r_{\text{sat}}$).
   - Medium star is still well detected above background.
   - Scale factor:
     $$k_{\text{MB}} = \text{median}\left(\frac{I_{\text{medium, scaled}}(r)}{I_{\text{bright}}(r)}\right)$$


In [5]:
# Overlap Zone 1: Faint <-> Medium
r_min_fm, r_max_fm = 10.0, 18.0
k_fm = rep.determine_overlap_scale_factor_dual(prof_f, prof_m, r_f, r_m, r_min=r_min_fm, r_max=r_max_fm)

# Scale Medium profile
prof_m_scaled = prof_m * k_fm
err_m_scaled = err_m * k_fm

# Overlap Zone 2: Scaled Medium <-> Bright
r_min_mb, r_max_mb = 25.0, 45.0
k_mb = rep.determine_overlap_scale_factor_dual(prof_m_scaled, prof_b, r_m, r_b, r_min=r_min_mb, r_max=r_max_mb)

# Scale Bright profile
prof_b_scaled = prof_b * k_mb
err_b_scaled = err_b * k_mb

print("Calculated Scalar Scale Factors:")
print(f"  k_FM (Faint-to-Medium) : {k_fm:.5f} (flux ratio ~ {1/k_fm:.2f}x)")
print(f"  k_MB (Medium-to-Bright): {k_mb:.5f} (flux ratio ~ {1/k_mb:.2f}x)")
print(f"  Total Dynamic Range Factor (Bright / Faint): {1.0 / (k_fm * k_mb):,.1f}x")


Calculated Scalar Scale Factors:
  k_FM (Faint-to-Medium) : 0.10805 (flux ratio ~ 9.26x)
  k_MB (Medium-to-Bright): 0.03519 (flux ratio ~ 28.42x)
  Total Dynamic Range Factor (Bright / Faint): 263.0x


## Step 5: High-Dynamic-Range Radial Profile Overlay

We visualize the raw and normalized radial profiles together with the shaded normalization regions and saturation boundary.

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Raw Unscaled Profiles
ax1.plot(r_b * l2.pixel_scale, prof_b, label=f"Bright (m={stamp_bright.mag:.2f})", color="#d62728", lw=2)
ax1.plot(r_m * l2.pixel_scale, prof_m, label=f"Medium (m={stamp_medium.mag:.2f})", color="#2ca02c", lw=2)
ax1.plot(r_f * l2.pixel_scale, prof_f, label=f"Faint  (m={stamp_faint.mag:.2f})", color="#1f77b4", lw=2)

ax1.axvspan(0, r_sat_px * l2.pixel_scale, color="#d62728", alpha=0.15, label="Bright Core Saturation")
ax1.axvspan(r_min_fm * l2.pixel_scale, r_max_fm * l2.pixel_scale, color="#2ca02c", alpha=0.2, label="Zone 1: Faint-Medium Overlap")
ax1.axvspan(r_min_mb * l2.pixel_scale, r_max_mb * l2.pixel_scale, color="#ff7f0e", alpha=0.2, label="Zone 2: Medium-Bright Overlap")

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Radius (arcseconds)", fontsize=12)
ax1.set_ylabel("Surface Brightness (MJy/sr)", fontsize=12)
ax1.set_title("1. Raw Unscaled Stellar Profiles", fontsize=13, fontweight="bold")
ax1.grid(True, which="both", alpha=0.3, ls="--")
ax1.legend(loc="upper right", fontsize=9)

# Plot 2: Scaled Coherent Profile
ax2.plot(r_f * l2.pixel_scale, prof_f, label="Faint (Core Reference)", color="#1f77b4", lw=2.5)
ax2.plot(r_m * l2.pixel_scale, prof_m_scaled, label=f"Medium (Scaled x{k_fm:.3f})", color="#2ca02c", lw=2, ls="--")
ax2.plot(r_b * l2.pixel_scale, prof_b_scaled, label=f"Bright (Scaled x{k_mb*k_fm:.3e})", color="#d62728", lw=1.8, ls=":")

ax2.axvspan(0, r_sat_px * l2.pixel_scale, color="#d62728", alpha=0.15, label="Saturated Core")
ax2.axvspan(r_min_fm * l2.pixel_scale, r_max_fm * l2.pixel_scale, color="#2ca02c", alpha=0.2, label="Overlap 1 [10-18 px]")
ax2.axvspan(r_min_mb * l2.pixel_scale, r_max_mb * l2.pixel_scale, color="#ff7f0e", alpha=0.2, label="Overlap 2 [25-45 px]")

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("Radius (arcseconds)", fontsize=12)
ax2.set_ylabel("Scaled Relative Surface Brightness", fontsize=12)
ax2.set_title("2. Seamlessly Normalized Composite Radial Profile", fontsize=13, fontweight="bold")
ax2.grid(True, which="both", alpha=0.3, ls="--")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()


/var/folders/l3/dvw93ttj6hnf38k5r_njqf800000gq/T/ipykernel_87086/1934782239.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6: 2D Spatial Visualization of Normalization Annuli

To verify that the normalization regions align with the physical optical features of each star, we superimpose the normalization circles and saturation boundaries directly onto the 2D star cutouts.

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

stamps = [
    (stamp_faint, "Faint Star (Core)", axes[0], [
        (r_min_fm, "#2ca02c", "--", "Zone 1 (inner)"),
        (r_max_fm, "#2ca02c", "-", "Zone 1 (outer)")
    ]),
    (stamp_medium, "Medium Star (Transition)", axes[1], [
        (r_min_fm, "#2ca02c", "--", "Zone 1 (inner)"),
        (r_max_fm, "#2ca02c", "-", "Zone 1 (outer)"),
        (r_min_mb, "#ff7f0e", "--", "Zone 2 (inner)"),
        (r_max_mb, "#ff7f0e", "-", "Zone 2 (outer)")
    ]),
    (stamp_bright, "Bright Star (Wings & Spikes)", axes[2], [
        (r_sat_px, "#d62728", "-", "Saturation Radius"),
        (r_min_mb, "#ff7f0e", "--", "Zone 2 (inner)"),
        (r_max_mb, "#ff7f0e", "-", "Zone 2 (outer)")
    ]),
]

for s_obj, title, ax, circles in stamps:
    d = s_obj.data
    sz = d.shape[0]
    cen = (sz - 1) / 2.0
    
    # Zoom in to central 160x160 px for visibility if bright stamp is 1001x1001
    zoom_radius = min(cen, 80)
    x0 = int(cen - zoom_radius)
    x1 = int(cen + zoom_radius + 1)
    d_sub = d[x0:x1, x0:x1]
    
    vmax = np.nanpercentile(d, 99.8)
    vmin = max(np.nanpercentile(d, 5), 1e-4)
    
    im = ax.imshow(np.maximum(d_sub, 1e-4), origin="lower", cmap="inferno", norm=LogNorm(vmin=vmin, vmax=vmax),
                   extent=[-zoom_radius, zoom_radius, -zoom_radius, zoom_radius])
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("$\Delta X$ (pixels)")
    ax.set_ylabel("$\Delta Y$ (pixels)")
    
    # Draw circles
    for rad, color, ls, lbl in circles:
        c = Circle((0, 0), rad, color=color, fill=False, lw=1.8, ls=ls, label=lbl)
        ax.add_patch(c)
        
    ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Annular Normalization & Saturation Boundaries Superimposed on 2D Stamps", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


<>:38: SyntaxWarning: invalid escape sequence '\D'
<>:39: SyntaxWarning: invalid escape sequence '\D'
<>:38: SyntaxWarning: invalid escape sequence '\D'
<>:39: SyntaxWarning: invalid escape sequence '\D'
/var/folders/l3/dvw93ttj6hnf38k5r_njqf800000gq/T/ipykernel_87086/8905931.py:38: SyntaxWarning: invalid escape sequence '\D'
  ax.set_xlabel("$\Delta X$ (pixels)")
/var/folders/l3/dvw93ttj6hnf38k5r_njqf800000gq/T/ipykernel_87086/8905931.py:39: SyntaxWarning: invalid escape sequence '\D'
  ax.set_ylabel("$\Delta Y$ (pixels)")
/var/folders/l3/dvw93ttj6hnf38k5r_njqf800000gq/T/ipykernel_87086/8905931.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7: Reusable Function to Profile & Analyze Any Custom Stamp

This modular function takes any 2D star stamp array and mask, measures its radial profile, locates its saturation radius, and plots the profile with normalization regions highlighted.

In [8]:
def analyze_custom_stamp(stamp_data, stamp_mask=None, star_label="Custom Star", pixel_scale=0.11):
    """
    Compute and plot the radial profile and normalization zones for any given star stamp.
    """
    ny, nx = stamp_data.shape
    rmax = (min(ny, nx) - 1) // 2
    
    # Measure 1D radial profile
    radii, flux, flux_err = rep.measure_radial_profile_1d(stamp_data, mask=stamp_mask, rmax=rmax)
    
    # Plot profile
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(radii * pixel_scale, flux, yerr=flux_err, fmt="-o", markersize=3, color="navy", ecolor="lightblue", lw=1.5, label="Measured Profile")
    
    # Highlight zones
    ax.axvspan(10 * pixel_scale, 18 * pixel_scale, color="green", alpha=0.18, label="Overlap Zone 1 (10-18 px)")
    ax.axvspan(25 * pixel_scale, 45 * pixel_scale, color="orange", alpha=0.18, label="Overlap Zone 2 (25-45 px)")
    
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Radius (arcseconds)", fontsize=11)
    ax.set_ylabel("Flux / Surface Brightness", fontsize=11)
    ax.set_title(f"Radial Profile & Normalization Zones: {star_label}", fontsize=12, fontweight="bold")
    ax.grid(True, which="both", alpha=0.3, ls="--")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()
    return radii, flux

# Example usage on the medium star stamp
r_out, f_out = analyze_custom_stamp(stamp_medium.data, stamp_medium.mask, star_label=f"Medium Star (m={stamp_medium.mag:.2f})")


/var/folders/l3/dvw93ttj6hnf38k5r_njqf800000gq/T/ipykernel_87086/1009278000.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary & Physical Implications

- **1D Profiles Are Strictly for Normalization**: As required by the Roman extended PSF methodology, 1D profiles are used **only** to identify saturation boundaries ($r_{\text{sat}}$) and calculate the scalar cross-tier scaling factors ($k_{\text{FM}}, k_{\text{MB}}$).
- **Zero Circularization in 2D**: The final 2D extended PSF canvas is assembled by smoothly blending the **2D image stamps directly** using partition-of-unity cosine weighting across the overlap zones. This guarantees that Roman's **6-fold optical diffraction spikes and spider asymmetry are 100% preserved in the final model**!
